In [ ]:
# import necessary packages
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole, rdMolDraw2D
from rdkit.Chem import AllChem
import math
from PIL import Image
import io

In [ ]:
# identify path to data
substituents = 'hammett_predictions.csv'

df = pd.read_csv(substituents)

In [ ]:
# drop duplicate substituents (substituents are listed twice in the dataframe, once for meta and once for para)
df = df.drop_duplicates(subset=['Substituent'])

# verify that we have the correct number of substituents
len(df)

In [ ]:
# Replace [At] with the dummy atom * so that RDKit can draw the substituents correctly
df['Substituent'] = df['Substituent'].str.replace('[At]', '[*]', regex=False)

df.head()

In [ ]:
# create mol objects for the substituents
smis = df['Substituent']
mol_objects = [Chem.MolFromSmiles(smi) for smi in smis]

# define the allowable size of one page for the figure
mols_per_row = 10
dpi = 400
max_height_inches = 9.1
max_height_px = int(max_height_inches * dpi)  # 3640 px

target_width_px = int(6.5 * dpi)  # 2600 px
subimg_w = target_width_px // mols_per_row
subimg_size = (subimg_w, subimg_w)

# set the RDKit drawing parameters
draw_options = rdMolDraw2D.MolDrawOptions()
draw_options.useBWAtomPalette()
draw_options.dummiesAreAttachments = True
draw_options.padding = 0.05
draw_options.bondLineWidth = 3

# determine the number of rows that can be on one page and the number of molecules that can be on one page
rows_per_page = max_height_px // subimg_w
mols_per_page = rows_per_page * mols_per_row
mols_first_page = mols_per_page - (mols_per_row * 5)  # five rows shorter

# allow for a variable number of rows on the first page, based on the formatting of the manuscript
total_mols = len(mol_objects)
remaining = total_mols - mols_first_page
num_pages = 1 + math.ceil(remaining / mols_per_page)

print(f"Molecules: {total_mols} | Rows per page: {rows_per_page} | First page rows: {rows_per_page - 1} | Pages: {num_pages}")

# draw the molecules and save the drawings to the necessary number of pngs, allowing for a variable number of rows on the first page
for page_idx in range(num_pages):
    if page_idx == 0:
        start, end = 0, min(mols_first_page, total_mols)
    else:
        start = mols_first_page + (page_idx - 1) * mols_per_page
        end = min(start + mols_per_page, total_mols)

    page_mols = mol_objects[start:end]

    img = Draw.MolsToGridImage(
        page_mols,
        molsPerRow=mols_per_row,
        subImgSize=subimg_size,
        drawOptions=draw_options,
        maxMols=len(page_mols),
        useSVG=False
    )

    pil_img = Image.open(io.BytesIO(img.data))
    filename = f"dataset_page{page_idx + 1:02d}.png"
    pil_img.save(filename, dpi=(dpi, dpi))
    print(f"Saved {filename} — {len(page_mols)} molecules, size {pil_img.size}")